In [2]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

/opt/conda/envs/incontext_spurious/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/envs/incontext_spurious/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/conda/envs/incontext_spurious/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vi

In [6]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('roberta-base')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
inputs = [
    ("You should probably", "TRAIN this model"),
    ("on a down-stream", "task to"),
    ("be able to use it for predictions", "and inference")
]

premises, hypotheses = zip(*inputs)

In [33]:
tokenizer(premises[0], hypotheses[0])

{'input_ids': [0, 1185, 197, 1153, 2, 2, 565, 4396, 2444, 42, 1421, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [35]:
tokenizer(premises[1], hypotheses[1])

{'input_ids': [0, 261, 10, 159, 12, 8656, 2, 2, 45025, 7, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [34]:
tokenizer(premises, hypotheses)

{'input_ids': [[0, 1185, 197, 1153, 2, 2, 565, 4396, 2444, 42, 1421, 2], [0, 261, 10, 159, 12, 8656, 2, 2, 45025, 7, 2], [0, 1610, 441, 7, 304, 24, 13, 12535, 2, 2, 463, 42752, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [40]:
from typing import List, Tuple


class RoBERTa(nn.Module):
    def __init__(self, 
            model_name, 
            *args, 
            **kwargs
        ):
        super().__init__()

        self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
        self.model = RobertaModel.from_pretrained(model_name)

    def forward(self, x: List[Tuple[str, str]]):

        premises, hypotheses = zip(*x)
        inputs = self.tokenizer(premises, hypotheses, return_tensors="pt", padding=True, truncation=True)

        inputs = {key: value.to(self.model.device) for key, value in inputs.items() if isinstance(value, torch.Tensor)}

        outputs = self.model(**inputs)

        sentence_embeddings = outputs.last_hidden_state[:, 0, :] # Shape: (batch_size, hidden_size)

        return sentence_embeddings


In [41]:
roberta = RoBERTa('roberta-base')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
ret = roberta(inputs)

In [44]:
ret.shape

torch.Size([3, 768])